# Клиенты ТЭ за июль 2026: три подсчёта

Зерно — договор. Три числа:

1. **final_df.** Актуальный файл: `Проверки в пространстве/final_script_2_1.ipynb`, функция `compute_final_df_for_month`, секция `01_sa_perimeter`. Старый `final_script_2.ipynb` не используется. Договор SA пересекает июль: открыт не позже 31.07 и закрыт не раньше 01.07 (или не закрыт). Привязка `P` тоже пересекает июль, но закрытие привязки строгое: `d_valid_to > 2026-07-01`. Дальше витрина только левыми джойнами добавляет атрибуты, число договоров не режет.
2. **Скрипт сборки ТЭ.** Файл `Проверки в пространстве/trade_acquiring_clients_month_excel.ipynb`, лист «С ТЭ». Договор и привязка `P` оба живы в конкретный день. Для июля этот день — 31.07.2026: `d_valid_to` пустая или `>= 2026-07-31`.
3. **Excel.** `/home/jovyan/documents/Equaring/Data/07_Июль_2026.xlsx`. Число — уникальные ID договора. Рядом печатается число строк файла.

Запустите ячейки по порядку. После сводки две проверки: 335 договоров final_df против среза на 31.07 и расхождение Excel с месячным периметром.


In [ ]:
import getpass

import pandas as pd
from rail_connectors.connection import connect

MONTH_START = '2026-07-01'
MONTH_END = '2026-07-31'
EXCEL_JUL = '/home/jovyan/documents/Equaring/Data/07_Июль_2026.xlsx'

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
sql_final_df = f'''
SELECT count(DISTINCT a.n_agr) AS contracts
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(cast(a.acq_class AS string))) = 'SA'
  AND cast(a.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
  AND (a.d_valid_to IS NULL OR cast(a.d_valid_to AS date) >= cast('{MONTH_START}' AS date))
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE cast(t.n_agr AS string) = cast(a.n_agr AS string)
      AND cast(t.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
      AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) > cast('{MONTH_START}' AS date))
      AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    n_final = int(imp.fetch(sql_final_df)['contracts'].iloc[0])
print('final_df, договоры за июль:', n_final)
n_final


In [ ]:
sql_script = f'''
SELECT count(DISTINCT a.n_agr) AS contracts
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(a.acq_class)) = 'SA'
  AND a.d_valid_from <= '{MONTH_END}'
  AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{MONTH_END}')
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE t.n_agr = a.n_agr
      AND t.d_valid_from <= '{MONTH_END}'
      AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{MONTH_END}')
      AND upper(trim(t.cf_ter_type)) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    n_script = int(imp.fetch(sql_script)['contracts'].iloc[0])
print('скрипт сборки ТЭ, договоры на 31.07.2026:', n_script)
n_script


In [ ]:
def norm_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s.endswith('.0'):
        s = s[:-2]
    if s in {'', 'nan', 'None'}:
        return None
    return s

def pick(columns, names):
    low = {str(c).replace('\n', ' ').strip().lower(): c for c in columns}
    for name in names:
        if name.lower() in low:
            return low[name.lower()]
    return None

raw = pd.read_excel(EXCEL_JUL, header=None, nrows=12)
header = 0
for i, row in raw.iterrows():
    cells = ' '.join(str(x).lower().replace('\n', ' ') for x in row.tolist() if pd.notna(x))
    if any(m in cells for m in ('аур', 'амортизац', 'фин. рез', 'фин.рез', 'фин рез')):
        header = i
        break
ex = pd.read_excel(EXCEL_JUL, header=header)
id_col = pick(ex.columns, ['ID договора', 'agr_id', 'abs_agr_id'])
num_col = pick(ex.columns, ['Номер договора'])
print('заголовок Excel, строка', header)
print('строк в файле:', len(ex))
if id_col:
    n_excel = int(ex[id_col].map(norm_id).nunique(dropna=True))
    print('уникальных ID договора (%s):' % id_col, n_excel)
elif num_col:
    n_excel = int(ex[num_col].map(norm_id).nunique(dropna=True))
    print('колонки ID договора нет, уникальных номеров (%s):' % num_col, n_excel)
else:
    n_excel = len(ex)
    print('колонки договора нет, беру число строк:', n_excel)
    print(list(ex.columns))
n_excel


In [ ]:
summary = pd.DataFrame([
    ['final_df, договоры за июль', n_final],
    ['скрипт сборки ТЭ, на 31.07.2026', n_script],
    ['Excel, уникальные договоры', n_excel],
], columns=['способ', 'договоров'])
summary


## Проверка 1. Лишние договоры июля на 31.07 уже не действуют

Гипотеза: разница final_df и скрипта — это договоры, которые пересекли июль, но на 31.07 уже не действуют.

Ячейка забирает те же два периметра, что счётчики выше. Гипотеза верна, если числа совпали со счётчиками, в скрипте нет договоров вне июля, разница равна списку «только final_df», и у каждой строки этого списка причина одна из двух: договор закрыт до 31.07 или на 31.07 договор ещё открыт, а привязка P уже закрыта.


In [ ]:
sql_month = f'''
SELECT DISTINCT
  cast(a.n_agr AS string) AS n_agr,
  cast(a.abs_agr_id AS string) AS agr_id,
  a.c_agr_number AS agr_num,
  cast(a.d_valid_from AS date) AS opened,
  cast(a.d_valid_to AS date) AS closed
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(cast(a.acq_class AS string))) = 'SA'
  AND cast(a.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
  AND (a.d_valid_to IS NULL OR cast(a.d_valid_to AS date) >= cast('{MONTH_START}' AS date))
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE cast(t.n_agr AS string) = cast(a.n_agr AS string)
      AND cast(t.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
      AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) > cast('{MONTH_START}' AS date))
      AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

sql_script_keys = f'''
SELECT DISTINCT
  cast(a.n_agr AS string) AS n_agr
FROM ods_alpha.scd1_agreements a
JOIN ods_alpha.scd1_companies c
  ON c.n_cmp = a.n_cmp_client
WHERE upper(trim(a.acq_class)) = 'SA'
  AND a.d_valid_from <= '{MONTH_END}'
  AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{MONTH_END}')
  AND coalesce(a.ods_deleted_flg, '0') <> '1'
  AND coalesce(c.ods_deleted_flg, '0') <> '1'
  AND c.c_inn IS NOT NULL
  AND EXISTS (
    SELECT 1
    FROM ods_alpha.scd1_agr_terms t
    WHERE t.n_agr = a.n_agr
      AND t.d_valid_from <= '{MONTH_END}'
      AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{MONTH_END}')
      AND upper(trim(t.cf_ter_type)) = 'P'
      AND coalesce(t.ods_deleted_flg, '0') <> '1'
  )
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    month_keys = imp.fetch(sql_month)
    script_keys = imp.fetch(sql_script_keys)

month_keys.columns = [c.lower() for c in month_keys.columns]
script_keys.columns = [c.lower() for c in script_keys.columns]
month_keys['n_agr'] = month_keys['n_agr'].map(norm_id)
month_keys = month_keys.drop_duplicates('n_agr')
script_ids = set(script_keys['n_agr'].map(norm_id).dropna())
month_ids = set(month_keys['n_agr'].dropna())
only_month_ids = month_ids - script_ids
only_script_ids = script_ids - month_ids

end = pd.Timestamp(MONTH_END)
gap = month_keys[month_keys['n_agr'].isin(only_month_ids)].copy()
gap['closed_dt'] = pd.to_datetime(gap['closed'])
gap['Причина'] = 'договор открыт на 31.07, привязка P уже закрыта'
gap.loc[gap['closed_dt'].notna() & (gap['closed_dt'] < end), 'Причина'] = 'договор закрыт до 31.07'

p_ids = gap.loc[gap['Причина'] != 'договор закрыт до 31.07', 'n_agr'].dropna().tolist()
p_info = pd.DataFrame(columns=['n_agr', 'p_open', 'p_closed'])
if p_ids:
    parts = []
    for i in range(0, len(p_ids), 200):
        inn_sql = ', '.join("'%s'" % x.replace("'", '') for x in p_ids[i:i + 200])
        sql_p = f'''
        SELECT
          cast(t.n_agr AS string) AS n_agr,
          max(CASE WHEN t.d_valid_to IS NULL THEN 1 ELSE 0 END) AS p_open,
          max(cast(t.d_valid_to AS date)) AS p_closed
        FROM ods_alpha.scd1_agr_terms t
        WHERE cast(t.n_agr AS string) IN ({inn_sql})
          AND cast(t.d_valid_from AS date) <= cast('{MONTH_END}' AS date)
          AND (t.d_valid_to IS NULL OR cast(t.d_valid_to AS date) > cast('{MONTH_START}' AS date))
          AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
          AND coalesce(t.ods_deleted_flg, '0') <> '1'
        GROUP BY cast(t.n_agr AS string)
        '''
        with imp:
            parts.append(imp.fetch(sql_p))
    p_info = pd.concat(parts, ignore_index=True)
    p_info.columns = [c.lower() for c in p_info.columns]
    p_info['n_agr'] = p_info['n_agr'].map(norm_id)

gap = gap.merge(p_info, on='n_agr', how='left')
gap = gap.rename(columns={
    'agr_num': 'Номер договора',
    'opened': 'Дата открытия',
    'closed': 'Дата закрытия',
    'p_open': 'P ещё открыта',
    'p_closed': 'Дата закрытия P',
})
gap = gap[['Номер договора', 'Дата открытия', 'Дата закрытия', 'Причина', 'P ещё открыта', 'Дата закрытия P']]
gap = gap.sort_values(['Причина', 'Номер договора'])

hypothesis_ok = (
    len(month_ids) == n_final
    and len(script_ids) == n_script
    and len(only_script_ids) == 0
    and len(only_month_ids) == n_final - n_script
)
verdict = 'гипотеза верна' if hypothesis_ok else 'гипотеза не сходится с числами выше, смотрите строки'
print('договоров final_df:', len(month_ids), '| счётчик ячейки:', n_final)
print('договоров скрипта:', len(script_ids), '| счётчик ячейки:', n_script)
print('только final_df, на 31.07 уже нет:', len(only_month_ids))
print('только скрипт, вне июля:', len(only_script_ids))
print('разрыв final_df - скрипт:', n_final - n_script)
print(gap.groupby('Причина').size())
print(verdict)
gap


## Проверка 2. Кто разводит Excel и final_df

Сверка с месячным периметром, не со срезом на 31.07. Ключ — тот, который совпал лучше: ID договора из Excel с abs_agr_id или с номером договора.

«Только Excel» — договор есть в отчёте и не вошёл в final_df. Причина считается по Альфе: класс, удаление, ИНН, даты договора, привязка P. «Только final_df» — договор есть в месячном периметре и этого ключа нет в Excel.


In [ ]:
if id_col is None and num_col is None:
    raise RuntimeError('В Excel нет колонки договора: %s' % list(ex.columns))

month_keys['agr_id'] = month_keys['agr_id'].map(norm_id)
month_keys['agr_num'] = month_keys['agr_num'].map(norm_id)
excel_inn_col = pick(ex.columns, ['ИНН', 'inn', 'c_inn'])
if id_col:
    excel_keys = ex[id_col].map(norm_id)
    key_name = id_col
else:
    excel_keys = ex[num_col].map(norm_id)
    key_name = num_col
excel_cmp = pd.DataFrame({'key': excel_keys})
if excel_inn_col:
    excel_cmp['inn'] = ex[excel_inn_col].map(norm_id)
else:
    excel_cmp['inn'] = None
excel_cmp = excel_cmp.dropna(subset=['key']).drop_duplicates('key')

id_hit = excel_cmp['key'].isin(set(month_keys['agr_id'].dropna())).sum()
num_hit = excel_cmp['key'].isin(set(month_keys['agr_num'].dropna())).sum()
use_num = num_hit > id_hit
lake_col = 'agr_num' if use_num else 'agr_id'
print('совпало с abs_agr_id:', int(id_hit))
print('совпало с номером договора:', int(num_hit))
print('ключ сверки:', 'номер договора' if use_num else 'abs_agr_id', '| колонка Excel:', key_name)

lake_keys = set(month_keys[lake_col].dropna())
excel_ids = set(excel_cmp['key'])
only_excel = excel_ids - lake_keys
only_lake = lake_keys - excel_ids
print('только Excel:', len(only_excel))
print('только final_df:', len(only_lake))
print('Excel - final_df:', n_excel - n_final)

def sql_in(values):
    return ', '.join("'%s'" % str(v).replace("'", '') for v in values)

only_list = sorted(only_excel)
agr = pd.DataFrame()
terms = pd.DataFrame()
if only_list:
    agr_parts = []
    for i in range(0, len(only_list), 200):
        ids_sql = sql_in(only_list[i:i + 200])
        where_key = "a.c_agr_number IN (%s)" % ids_sql if use_num else "cast(a.abs_agr_id AS string) IN (%s)" % ids_sql
        sql_agr = f'''
        SELECT
          cast(a.n_agr AS string) AS n_agr,
          cast(a.abs_agr_id AS string) AS agr_id,
          a.c_agr_number AS agr_num,
          upper(trim(cast(a.acq_class AS string))) AS acq_class,
          cast(a.d_valid_from AS date) AS opened,
          cast(a.d_valid_to AS date) AS closed,
          a.ods_deleted_flg AS agr_deleted,
          c.c_inn AS inn,
          c.ods_deleted_flg AS cmp_deleted
        FROM ods_alpha.scd1_agreements a
        LEFT JOIN ods_alpha.scd1_companies c
          ON c.n_cmp = a.n_cmp_client
        WHERE {where_key}
        '''
        with imp:
            agr_parts.append(imp.fetch(sql_agr))
    agr = pd.concat(agr_parts, ignore_index=True)
    agr.columns = [c.lower() for c in agr.columns]
    agr['agr_id'] = agr['agr_id'].map(norm_id)
    agr['agr_num'] = agr['agr_num'].map(norm_id)
    agr['n_agr'] = agr['n_agr'].map(norm_id)
    n_list = sorted(set(agr['n_agr'].dropna()))
    term_parts = []
    for i in range(0, len(n_list), 200):
        ids_sql = sql_in(n_list[i:i + 200])
        sql_terms = f'''
        SELECT
          cast(t.n_agr AS string) AS n_agr,
          upper(trim(cast(t.cf_ter_type AS string))) AS ter_type,
          cast(t.d_valid_from AS date) AS opened,
          cast(t.d_valid_to AS date) AS closed,
          t.ods_deleted_flg AS deleted
        FROM ods_alpha.scd1_agr_terms t
        WHERE cast(t.n_agr AS string) IN ({ids_sql})
        '''
        with imp:
            term_parts.append(imp.fetch(sql_terms))
    if term_parts:
        terms = pd.concat(term_parts, ignore_index=True)
        terms.columns = [c.lower() for c in terms.columns]
        terms['n_agr'] = terms['n_agr'].map(norm_id)

start = pd.Timestamp(MONTH_START)
end = pd.Timestamp(MONTH_END)

def deleted(v):
    return str(v).strip() in {'1', '1.0'}

def why_missing(key):
    if agr.empty:
        return 'в Альфе нет договора с этим ключом', None
    rows = agr[agr['agr_num' if use_num else 'agr_id'] == key]
    if rows.empty:
        return 'в Альфе нет договора с этим ключом', None
    sa = rows[rows['acq_class'] == 'SA']
    row = (sa if len(sa) else rows).iloc[0]
    info = row
    if row['acq_class'] != 'SA':
        return 'класс не SA: %s' % row['acq_class'], info
    if deleted(row['agr_deleted']):
        return 'договор удалён', info
    inn = row['inn']
    if pd.isna(inn) or str(inn).strip() == '':
        return 'нет ИНН', info
    if deleted(row['cmp_deleted']):
        return 'компания удалена', info
    opened = pd.to_datetime(row['opened'])
    closed = pd.to_datetime(row['closed']) if pd.notna(row['closed']) else pd.NaT
    if pd.notna(opened) and opened > end:
        return 'открыт после 31.07', info
    if pd.notna(closed) and closed < start:
        return 'закрыт до 01.07', info
    t = terms[terms['n_agr'] == row['n_agr']] if len(terms) else terms
    live = t[~t['deleted'].map(deleted)] if len(t) else t
    if live.empty:
        return 'нет живой привязки agr_terms', info
    p = live[live['ter_type'] == 'P']
    if p.empty:
        kinds = ', '.join(sorted(set(live['ter_type'].dropna().astype(str))))
        return 'нет привязки P, есть типы: %s' % (kinds or 'пусто'), info
    p_open = pd.to_datetime(p['opened'])
    p_close = pd.to_datetime(p['closed'])
    overlaps = (
        (p_open <= end)
        & (p_close.isna() | (p_close > start))
    )
    if overlaps.any():
        return 'привязка P пересекает июль, ключ отчёта не совпал с периметром', info
    if (p_close.notna() & (p_close <= start)).any() and not (p_open > end).any():
        return 'привязка P закрыта не позже 01.07', info
    if (p_open > end).any():
        return 'привязка P открыта после 31.07', info
    return 'привязка P не пересекает июль', info

miss_rows = []
for key in sorted(only_excel):
    reason, info = why_missing(key)
    inn_excel = excel_cmp.loc[excel_cmp['key'] == key, 'inn']
    miss_rows.append({
        'Ключ': key,
        'ИНН в Excel': None if inn_excel.empty else inn_excel.iloc[0],
        'Причина': reason,
        'Класс': None if info is None else info['acq_class'],
        'Дата открытия': None if info is None else info['opened'],
        'Дата закрытия': None if info is None else info['closed'],
        'Номер договора': None if info is None else info['agr_num'],
    })
excel_miss = pd.DataFrame(miss_rows)
if len(excel_miss):
    print('почему нет в final_df:')
    print(excel_miss.groupby('Причина').size())

lake_miss = month_keys[month_keys[lake_col].isin(only_lake)][['agr_id', 'agr_num', 'opened', 'closed']].copy()
lake_miss = lake_miss.rename(columns={
    'agr_id': 'ID договора',
    'agr_num': 'Номер договора',
    'opened': 'Дата открытия',
    'closed': 'Дата закрытия',
})
lake_miss = lake_miss.sort_values('Номер договора')
print('в final_df и нет в Excel:', len(lake_miss))
display(excel_miss)
lake_miss
